# mjo_wavenum_freq_season

- "Calculates wavenumber-frequency spectra via seasonal averaging as defined by the US-CLIVAR MJO diagnostics website"
- [NCL Reference](https://www.ncl.ucar.edu/Document/Functions/Diagnostics/mjo_wavenum_freq_season.shtml)

### Example NCL Script and Output

- mjo_wavenum_freq_season.ncl
- mjo_output/mjo_wavenum_freq_season_output.txt
  
```{literalinclude} mjo_wavenum_freq_season.ncl

```

MJO CLIVAR: Wave number-frequency spectra
- "winter": 180 days (starts November 1)
- "summer": 180 days (starts May 1)
- "summer": 365 days (starts January 1)

In [1]:
import xarray as xr
import os
from datetime import datetime

In [2]:
u850_data = xr.open_dataset(os.getcwd() + "/data/anomaly/QBOi.EXP1.AMIP.001.u850.day.anom.nc")
u850_data

<xarray.Dataset> Size: 565MB
Dimensions:  (time: 2555, lat: 192, lon: 288)
Coordinates:
  * time     (time) object 20kB 1975-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.3 357.5 358.8
Data variables:
    date     (time) float64 20kB ...
    U850     (time, lat, lon) float32 565MB ...

[MJO wavenumber-frequency spectra based on Level 2 diagnostics](- Based on [US-CLIVAR MJO Working Group (2009) "MJO Simulation Diagnostics - Level 2 diagnostics"](https://doi.org/10.1175/2008JCLI2731.1))

> "Level 2 diagnostics are designed to explore more detailed features of the MJO. They include wavenumber-frequency spectra of individual fields, cross-spectral quantities between different fields, and a multivariate EOF analysis. Wavenumber-frequency spectra for equatorial precipitation and 850-hPa zonal wind are shown in Fig. 7 for boreal summer, and in Fig. 8 for boreal winter.

> The spectra were computed by Fourier transforming 180-day segments centered on boreal summer and boreal winter, forming power, and then averaging over all years of data (1979–2005). The resulting bandwidth is (180 days)−1. Only the climatological season cycle was removed before calculation of the spectra.

> By definition, eastward propagation is represented by positive frequency and positive wavenumber whereas westward propagation is represented with one or the other of the frequency or wavenumber being negative. If standing oscillations are present, they will project as equal amounts of power in eastward and westward directions. The results indicate a concentration of power at 30–90-day periods and zonal wavenumber 1 for 850-hPa zonal wind, and zonal wavenumbers 1–3 for precipitation and OLR (e.g., Salby and Hendon 1994)"

In [28]:
def mjo_wavenum_freq_season_as_python(input_data, data_var, seasonName):
    # Python equivalent of mjo_wavenum_freq_season()
    data = input_data[data_var]

    seasonName = seasonName.lower()
    if seasonName == "winter":
        date_yyyymmdd = datetime(2025, 11, 1)
        ndays = 180
    if seasonName == "summer":
        date_yyyymmdd = datetime(2025, 5, 1)
        ndays = 180
    if seasonName == "annual":
        date_yyyymmdd = datetime(2025, 1, 1)
        ndays = 365
    
    print(f"Date for {seasonName}: {date_yyyymmdd}")
    print(data.groupby("time.season")) # group by seasons


    # remove seasonal cycle from data before apply 2D FFT
    ## calculate mean seasonal climate for each season across time range
    seasonal_climatology_mean = data.groupby("time.season").mean("time", skipna=True)
    ## remove seasonal cycle (deseasonalize) from data
    data_deseason = data.groupby("time.season") - seasonal_climatology_mean

    print(data_deseason.coords)
    n_time = len(data_deseason["time"])
    m_lon = len(data_deseason["lon"])
    print(f"Dimensions (time x space): {n_time} (time) x {m_lon} (lon)")

    # average Wavenumber-Frequency power spectra over seasons via 2D FFT
    import numpy as np
    time_step_dt = 180 # daily
    lon_step_dx = 1 # degrees longitude (1 spatial step)

    ## 2D FFT
    spectrum = np.fft.fft2(data_deseason)
    ## Noramlize power spectrum
    power_spectral_density = np.abs(spectrum)**2 / (n_time * m_lon)
    ## Determine frequency
    freqs = np.fft.fftfreq(n_time, d=time_step_dt)
    ## Determine wavenumbers
    wavenumbers = np.fft.fftfreq(m_lon, d=lon_step_dx)
    ## Shift the zero frequency and wavenumber to the center
    psd_shifted = np.fft.fftshift(power_spectral_density)
    freqs_shifted = np.fft.fftshift(freqs)
    wavenumbers_shifted = np.fft.fftshift(wavenumbers)

    print(f"[wave | {len(wavenumbers_shifted)}] x [freq | {len(freqs_shifted)}]")
    
    # X = Zonal Wavenumber (cycles/dx)
    # Y = Frequency (cycles/dt)

In [29]:
mjo_wavenum_freq_season_as_python(u850_data, "U850", "winter")

Date for winter: 2025-11-01 00:00:00
<DataArrayGroupBy, grouped over 1 grouper(s), 4 groups in total:
    'season': UniqueGrouper('season'), 4/4 groups with labels 'DJF', 'JJA', 'MAM', 'SON'>
Coordinates:
  * time     (time) object 20kB 1975-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.3 357.5 358.8
    season   (time) object 20kB 'DJF' 'DJF' 'DJF' 'DJF' ... 'DJF' 'DJF' 'DJF'
Dimensions (time x space): 2555 (time) x 288 (lon)
[wave | 288] x [freq | 2555]


# mjo_wavenum_freq_season_plot

- "Plot wavenumber-frequency spectra as returned by mjo_wavenum_freq_season"
- [NCL Reference](https://www.ncl.ucar.edu/Document/Functions/Diagnostics/mjo_wavenum_freq_season_plot.shtml)

### Example Plot fom `mjo_wavenum_freq_season.ncl`

- mjo_output/mjo_wavenum_freq_season_plot.winter.png

<center>
<img src="mjo_output/mjo_wavenum_freq_season_plot.winter.png" width="400" height="600">
</center>

In [13]:
def mjo_wavenum_freq_season_plot_as_python(dt, wavenumbers_shifted, freqs_shifted, psd_shifted):
    ## TODO: WIP
    import matplotlib.pyplot as plt

    plt.figure(figsize=(10, 6))
    # Use pcolormesh for 2D data
    plt.pcolormesh(wavenumbers_shifted, freqs_shifted, np.log10(psd_shifted), shading='auto') 
    plt.colorbar(label='log10(Power Spectral Density)')
    plt.xlabel('Zonal Wavenumber (cycles/dx)')
    plt.ylabel('Frequency (cycles/dt)')
    plt.title(f'Wavenumber-Frequency Spectrum for Summer (JJA)')
    plt.xlim([-10, 10]) # Limit to relevant wavenumbers
    plt.ylim([0, 0.5/dt]) # Limit to positive frequencies up to Nyquist
    plt.show()